# 七品训练与评估（1.0）
先运行环境与数据检查，再手动打开训练开关。CPU 基线与 CUDA 神经网络分开运行；参数、缩放器和结果均存本地。

当前数据为回顾研究候选，目标是 **七日后的来源参考价/收益**，不是可执行卖价或净利润。旧 test 已看过，只能探索；本 Notebook 默认不评分 test。四种神经网络同用过去 672 小时和 21 个特征。

第一次按顺序：运行全部（检查）→ CPU 开关设 True（运行 CPU 单元格）→ 评估开关设 True（运行评估和画图）→ 需要时关闭 CPU 开关，打开 CUDA 开关。运行后把开关改回 False。

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import display, Markdown, Image
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/train_seven_models.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("请在 CS_YouOweMe 项目内打开 Notebook")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.research.evaluation import seven_training as workflow
print(json.dumps(workflow.environment(), ensure_ascii=False, indent=2))
print("当前内核：", sys.executable)


你的现有环境已测可用：Python 3.11、torch 2.0.1+cu118、CUDA 11.8、RTX 3060 Laptop 6GB。内核要选同一 Python。不要在 Notebook 自动安装或替换 CUDA 包；缺依赖时参考项目使用说明。

In [ ]:
DATASET_ID = "seven_daily_20261020_v1"
PREPARED_ID = "seven_prep_20261021_v1"   # 已准备；新数据必须新建 ID
EXPERIMENT_ID = None                    # 训练后自动填；重启后可贴回历史 ID
REPORT_ID = None                        # 评估后自动填；重启后可贴回历史 ID
RUN_PREPARE = False
RUN_CPU_TRAIN = False
RUN_CUDA_TRAIN = False
RUN_EVALUATE = False
RUN_EXPLORATORY_TEST = False
CPU_MODELS = ["persistence", "ridge", "auto_arima", "lightgbm"]
CUDA_MODELS = ["persistence", "dlinear", "lstm", "transformer", "gated_lstm_transformer"]
CUDA_DEVICE = "cuda"                    # 显式 CUDA 不会静默回退 CPU
CUDA_OVERRIDES = {name: {"batch_size": 8, "epochs": 50, "patience": 8} for name in CUDA_MODELS if name != "persistence"}
if RUN_CPU_TRAIN and RUN_CUDA_TRAIN:
    raise ValueError("一次只打开一个训练开关，分别保存实验 ID")


In [ ]:
if RUN_PREPARE:
    prepared = workflow.prepare(DATASET_ID, prepared_id=PREPARED_ID)
else:
    try:
        _, prepared = workflow.verify(workflow.BASE / "prepared", PREPARED_ID, "prepared")
    except FileNotFoundError:
        prepared = None
        print("尚无预处理版本；将 RUN_PREPARE=True 后运行此格。已有版本不覆盖。")
if prepared:
    display({"prepared_id": PREPARED_ID, "splits": prepared["split_counts"], "source": prepared["dataset_id"]})
    print("只在 train 拟合 median/mean/std；val/test 不更新；原清洗库保持 NULL。")


CPU 训练：持平、Ridge、有界 ARIMA 和 LightGBM。ARIMA 固定 4 个阶数、只用训练价格按 AIC 选阶；不是无限搜索。每个模型结束后写进度，失败在 manifest 中保留，不自动替换。训练若中断，以新实验 ID 重跑；这是训练实验版本管理，采集续传机制另管。

In [ ]:
if RUN_CPU_TRAIN:
    trained = workflow.train(PREPARED_ID, CPU_MODELS, device="cpu")
    EXPERIMENT_ID = trained["artifact_id"]
    display(trained)
    print("请保存这个 EXPERIMENT_ID：", EXPERIMENT_ID)
else:
    print("CPU 训练关闭")


CUDA 训练：DLinear、LSTM、小 Transformer、带门控的 LSTM–Transformer cross attention，加持平作对照。训练 MSE、验证 MAE 早停；最佳权重存 `.pt`。这是新任务的改写版，不加载毕设的交通数据权重。6GB 显存默认 batch=8；若显存不足，确认失败后用 batch=4 或 2 开新实验，程序不会擅自换设备。

各轮耗时会打印。第一次可把 epochs 改为 2 验证自己的环境，再另开 50 epoch 正式实验；短轮不用于模型优劣结论。

In [ ]:
if RUN_CUDA_TRAIN:
    trained = workflow.train(PREPARED_ID, CUDA_MODELS, device=CUDA_DEVICE, overrides=CUDA_OVERRIDES)
    EXPERIMENT_ID = trained["artifact_id"]
    display(trained)
    print("请保存这个 EXPERIMENT_ID：", EXPERIMENT_ID)
else:
    print("CUDA 训练关闭")


评估只加载已保存的模型和预处理器，不重新 fit。默认对照 validation；模型选择依据验证收益 MAE，另报告价格 MAE/RMSE、各品误差。没有拟合不确定性区间；旧测试也不能作为全新独立证据。

In [ ]:
if RUN_EVALUATE:
    if not EXPERIMENT_ID:
        raise ValueError("先训练，或把本地已完成的 EXPERIMENT_ID 填入配置")
    evaluation = workflow.evaluate(EXPERIMENT_ID, split="validation", device=CUDA_DEVICE if RUN_CUDA_TRAIN else "auto")
    REPORT_ID = evaluation["artifact_id"]
    display(evaluation["scores"])
    print("请保存这个 REPORT_ID：", REPORT_ID)
if RUN_EXPLORATORY_TEST:
    if not EXPERIMENT_ID:
        raise ValueError("需要已完成的 EXPERIMENT_ID")
    exploratory = workflow.evaluate(EXPERIMENT_ID, split="test_exploratory", allow_exploratory_test=True)
    print("旧 test 仅探索：", exploratory["artifact_id"])


In [ ]:
if REPORT_ID:
    report_path, evaluation = workflow.verify(workflow.REPORTS, REPORT_ID, "evaluation")
    display(Markdown((report_path / "report.md").read_text(encoding="utf-8")))
    for chart in evaluation["charts"]:
        display(Markdown("**" + chart["item"] + "**"))
        display(Image(filename=str(report_path / chart["path"])))
else:
    print("训练并评估后，这里显示七品真实目标 / 七日前预测 / 持平与误差图。缺日期保持断线。")
if EXPERIMENT_ID:
    experiment_path, experiment = workflow.verify(workflow.BASE / "experiments", EXPERIMENT_ID, "experiment")
    import matplotlib.pyplot as plt
    for model in experiment["models"]:
        history_file = experiment_path / model / "history.json"
        if history_file.is_file():
            history = json.loads(history_file.read_text(encoding="utf-8"))
            fig, ax = plt.subplots(figsize=(8, 3))
            ax.plot([r["epoch"] for r in history], [r["validation_return_mae"] for r in history], label="validation return MAE")
            ax.set(title=model, xlabel="epoch", ylabel="return MAE"); ax.legend(); ax.grid(alpha=.2)
            display(fig); plt.close(fig)


保存位置：
- `data/local/artifacts/seven_training/prepared/<ID>/`：train-only 预处理参数、规范、来源哈希、准确代码。
- `data/local/artifacts/seven_training/experiments/<ID>/`：最佳参数、模型配置、训练曲线、环境、验证选择、源码与哈希。
- `data/local/reports/seven_training/<ID>/`：重载评估、逐行预测、各品误差和 PNG。

这些目录不上传 GitHub。Notebook 不是参数存储位置，关掉 VS Code 不影响已保存结果。下一步是检查训练结果与前向验证；UU 匹配报价、费用和实际解锁仍需接入，不能仅凭参考价预测自动下单。